# Hippocampus -> Amygdala  Find Communication Subspace

Reduced-Rank Regression communication-subspace analysis on cached HFA. Mirrors the upstream `find-communication-subspace.ipynb` (Binish/Semedo).

Two epoch sub-runs (encoding-cat / delay-load) x three configurations (CS / Within / Reverse). Per (sub-run, config): ridge-regularised RRR with CV across alpha and rank, 1-SE rule for picking CS dim, bootstrapped over channel subsets.

**Run after** `hipp_amyg_compute_hfa.ipynb`. Switch bands by editing `CONFIG["analysis_band"]`.


## 1. Setup & Config

In [ ]:
# Pin each worker's BLAS to a single thread BEFORE numpy/scipy/sklearn import.
import os
for _v in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
           'BLIS_NUM_THREADS', 'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(_v, '1')

import warnings, time, pickle
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from joblib import Parallel, delayed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats as scistats

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
warnings.filterwarnings('ignore', category=RuntimeWarning)


In [ ]:
PROJECT_ROOT = Path('.').resolve()
OUT_DIR = PROJECT_ROOT / 'outputs' / 'hipp_amyg_HFA_CSA_v2'
HFA_CACHE_DIR = OUT_DIR / 'hfa_cache'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Test-mode toggle for the RRR analysis.
# True  = fast/noisy (fewer bootstrap iters, fewer folds, narrower alpha grid).
# False = strict / paper-grade settings.
TEST_MODE = True

CONFIG = {
    'random_seed': 20260508,

    # --- Region pair (plot labels only) ---
    'source_region': 'Hippocampus',
    'target_region': 'Amygdala',

    # --- Multi-band selector ---
    'analysis_band': 'theta',

    # --- Trial grouping (encoding-cat sub-run) ---
    'category_keep': (1, 2, 3, 4, 5),

    # --- RRR (paper-grade defaults; bootstrap with replacement) ---
    'rrr_n_chans':         8,                          # ceiling for the rank grid; #ch sampled per pool
    'rrr_n_iters':         1000,
    'rrr_n_folds':         10,
    'rrr_alpha_init_grid': np.linspace(0.05, 1000, 100),  # passed to sklearn RidgeCV

    # --- Threading (RRR runner is I/O-light, threads are fine) ---
    'n_workers': 24,
}

# --- TEST_MODE override ---
if TEST_MODE:
    CONFIG['rrr_n_iters']         = 100
    CONFIG['rrr_n_folds']         = 5
    CONFIG['rrr_alpha_init_grid'] = np.linspace(0.05, 100, 20)
    print('[TEST_MODE override applied] '
          f"rrr_n_iters={CONFIG['rrr_n_iters']}, "
          f"rrr_n_folds={CONFIG['rrr_n_folds']}, "
          f"|alpha_init_grid|={len(CONFIG['rrr_alpha_init_grid'])}")

print(f"Source -> Target: {CONFIG['source_region']} -> {CONFIG['target_region']}")
print(f"Analysis band: {CONFIG['analysis_band']}")
print(f"Cache dir: {HFA_CACHE_DIR}/{CONFIG['analysis_band']}")
print(f"RRR: n_chans={CONFIG['rrr_n_chans']} (with replacement), "
      f"n_iters={CONFIG['rrr_n_iters']}, n_folds={CONFIG['rrr_n_folds']}, "
      f"threads={CONFIG['n_workers']}")


## 2. Helpers

In [29]:
# --- Trial grouping ---
def group_trials_by_load(trial_table):
    if 'loads' not in trial_table.columns:
        raise ValueError("trial_table needs a 'loads' column")
    load_values = sorted(trial_table['loads'].dropna().unique())
    groups, labels, counts = [], [], []
    for lv in load_values:
        idx = np.where(trial_table['loads'].values == lv)[0]
        groups.append(idx)
        labels.append(f'Load {int(lv) if lv == int(lv) else lv}')
        counts.append(len(idx))
    return {'groups': groups, 'group_labels': labels, 'group_counts': counts,
            'load_values': load_values,
            'summary': '\n'.join([f'Load grouping: {len(trial_table)} trials'] +
                                  [f'  {l}: {c}' for l, c in zip(labels, counts)])}


def group_trials_by_first_stim_category(trial_table, keep=(1, 2, 3, 4, 5)):
    if 'PicIDs_Encoding1' not in trial_table.columns:
        raise ValueError("trial_table needs a 'PicIDs_Encoding1' column")
    pic_ids = trial_table['PicIDs_Encoding1'].fillna(0).astype(int)
    cats = pic_ids // 100
    keep = list(keep)
    groups = [np.where(cats.values == c)[0] for c in keep]
    labels = [f'cat {c}' for c in keep]
    counts = [len(g) for g in groups]
    dropped = ~cats.isin(keep) | (pic_ids == 0)
    return {'groups': groups, 'group_labels': labels, 'group_counts': counts,
            'load_values': keep, 'dropped_trials': np.where(dropped.values)[0],
            'summary': '\n'.join([f'Category grouping: {len(trial_table)} trials, keep={tuple(keep)}'] +
                                  [f'  {l}: {c}' for l, c in zip(labels, counts)] +
                                  [f'  [dropped] {int(dropped.sum())} trials'])}


def get_trial_groups(trial_table, mode='category', keep=(1, 2, 3, 4, 5)):
    if mode == 'load':
        return group_trials_by_load(trial_table)
    if mode == 'category':
        return group_trials_by_first_stim_category(trial_table, keep=keep)
    raise ValueError(f"unknown trial_grouping mode: {mode!r}")


# --- Block aggregation ---
def _aggregate(X, factor, method='block_avg'):
    if factor is None or factor <= 1:
        return X
    if method == 'decimate':
        return X[::factor]
    if method == 'block_avg':
        n = (X.shape[0] // factor) * factor
        return X if n == 0 else X[:n].reshape(-1, factor, X.shape[1]).mean(axis=1)
    raise ValueError(f'unknown method {method!r}')


print('grouping + aggregation helpers defined')

grouping + aggregation helpers defined


In [30]:
# --- RRR aligned with upstream Binish/Semedo (commsub_utils.ReducedRankRegress).
#     Per outer fold: sklearn.linear_model.RidgeCV picks alpha by inner CV;
#     PCA on Y_hat = X @ B_full gives the rank projection; NSE = RSS/TSS scored
#     on test fold across ranks; 1-SE rule selects CS dim. R^2 = 1 - NSE is
#     reported for plot readability while internal arithmetic uses NSE.
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold


def build_rrr_data_matrix(tensor, groups):
    """Condition-average across trials within each group, then concatenate the
    per-condition time courses along the time axis.

    Returns shape (sum_n_t, n_chan): for each channel, a single time course
    formed by laying every condition's trial-averaged trace end to end. No
    downsampling here so RRR sees the data at the cached HFA rate."""
    pieces = []
    for grp in groups:
        if len(grp) == 0:
            continue
        cond_avg = tensor[grp].mean(axis=0).T.astype(np.float64)   # (n_t, n_chan)
        pieces.append(cond_avg)
    if not pieces:
        return np.zeros((0, tensor.shape[1]), dtype=np.float64)
    return np.concatenate(pieces, axis=0)


def cv_rrr_nse(X, Y, rank_grid, alpha_init_grid, n_folds, rng, iter_idx=0,
                ridge_inner_cv=10):
    """Outer K-fold CV. Per fold: RidgeCV (inner CV) picks alpha; PCA on Y_hat
    yields rank-projection; NSE = RSS/TSS scored per rank on the test fold.

    Returns nse: (n_folds, len(rank_grid)). Lower is better; 0 = perfect."""
    rank_grid = np.asarray(rank_grid, dtype=int)
    # KFold(shuffle=True, random_state=iter_idx) matches upstream's pattern.
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=int(iter_idx))
    nse = np.full((n_folds, len(rank_grid)), np.nan)
    for fi, (tr, te) in enumerate(kf.split(X)):
        Xtr, Xte = X[tr], X[te]
        Ytr, Yte = Y[tr], Y[te]

        # Center using train means
        x_mean = Xtr.mean(axis=0)
        y_mean = Ytr.mean(axis=0)
        Xtr_c = Xtr - x_mean
        Ytr_c = Ytr - y_mean
        Xte_c = Xte - x_mean

        # RidgeCV picks alpha by inner CV. fit_intercept=False because we
        # already centered both train sets.
        try:
            ridge = RidgeCV(alphas=np.asarray(alpha_init_grid, dtype=float),
                             cv=ridge_inner_cv, fit_intercept=False)
            ridge.fit(Xtr_c, Ytr_c)
        except Exception:
            continue
        B_full = ridge.coef_.T   # (p, K)

        # PCA on Y_hat_train via SVD on (Xtr_c @ B_full)
        Y_hat_tr = Xtr_c @ B_full
        try:
            _, _, Vt = np.linalg.svd(Y_hat_tr, full_matrices=False)
        except np.linalg.LinAlgError:
            continue
        V = Vt.T   # right singular vectors of Y_hat_train

        # TSS on test (variance of Yte from its own mean)
        ss_tot = float(((Yte - Yte.mean(axis=0)) ** 2).sum()) + 1e-12

        # Per-rank NSE
        for ri, m in enumerate(rank_grid):
            m_clip = min(int(m), V.shape[1])
            Vm = V[:, :m_clip]
            B_m = B_full @ Vm @ Vm.T
            # Reconstruct prediction in original scale
            Y_pred = (Xte_c @ B_m) + y_mean
            ss_res = float(((Yte - Y_pred) ** 2).sum())
            nse[fi, ri] = ss_res / ss_tot
    return nse


def select_dim_1se(nse_per_fold, rank_grid):
    """Smallest rank within 1 SE of the minimum mean NSE (Binish 1-SE rule).
    Returns (dim, mean_nse_curve, sem_nse_curve)."""
    rank_grid = np.asarray(rank_grid, dtype=int)
    mean_nse = np.nanmean(nse_per_fold, axis=0)
    n_valid = np.maximum(np.sum(~np.isnan(nse_per_fold), axis=0), 1)
    sem_nse = np.nanstd(nse_per_fold, axis=0, ddof=1) / np.sqrt(n_valid)
    if not np.any(np.isfinite(mean_nse)):
        return np.nan, mean_nse, sem_nse
    min_idx = int(np.nanargmin(mean_nse))
    threshold = mean_nse[min_idx] + sem_nse[min_idx]
    valid = (mean_nse <= threshold) & np.isfinite(mean_nse)
    if not valid.any():
        return np.nan, mean_nse, sem_nse
    return int(rank_grid[int(np.argmax(valid))]), mean_nse, sem_nse


def bootstrap_rrr(X_pool, Y_pool, n_chans, n_iters,
                   alpha_init_grid, rank_grid, n_folds, rng,
                   sampling='replace'):
    """Bootstrap RRR over channel subsets sampled WITH replacement.

    sampling='replace':  X = n_chans channels of X_pool (with replacement);
                          Y = n_chans channels of Y_pool (with replacement).
    sampling='disjoint': split X_pool channels once into two non-overlapping
                          halves (Within config); X = n_chans with replacement
                          from half A, Y = n_chans with replacement from half B.
    """
    rank_grid = np.asarray(rank_grid, dtype=int)
    n_x_total, n_y_total = X_pool.shape[1], Y_pool.shape[1]

    res = {
        'cs_dim':     np.full(n_iters, np.nan),
        'best_nse':   np.full(n_iters, np.nan),
        'best_r2':    np.full(n_iters, np.nan),
        'nse_curves': np.full((n_iters, len(rank_grid)), np.nan),
        'r2_curves':  np.full((n_iters, len(rank_grid)), np.nan),
        'sem_curves': np.full((n_iters, len(rank_grid)), np.nan),
        'rank_grid':  rank_grid,
    }
    for it in range(n_iters):
        if sampling == 'disjoint':
            half = n_x_total // 2
            if half < 1:
                continue
            perm = rng.permutation(n_x_total)
            half_a, half_b = perm[:half], perm[half:half + half]
            x_idx = rng.choice(half_a, n_chans, replace=True)
            y_idx = rng.choice(half_b, n_chans, replace=True)
        else:
            x_idx = rng.choice(n_x_total, n_chans, replace=True)
            y_idx = rng.choice(n_y_total, n_chans, replace=True)
        Xb, Yb = X_pool[:, x_idx], Y_pool[:, y_idx]
        try:
            nse_grid = cv_rrr_nse(Xb, Yb, rank_grid, alpha_init_grid,
                                    n_folds=n_folds, rng=rng, iter_idx=it)
            cs, mn, se = select_dim_1se(nse_grid, rank_grid)
            res['cs_dim'][it] = cs
            res['nse_curves'][it] = mn
            res['sem_curves'][it] = se
            res['r2_curves'][it]  = 1.0 - mn
            if np.any(np.isfinite(mn)):
                res['best_nse'][it] = float(np.nanmin(mn))
                res['best_r2'][it]  = float(1.0 - res['best_nse'][it])
        except Exception:
            continue
    return res


print('RRR helpers defined (RidgeCV + NSE + shuffled K-fold, upstream-aligned)')


RRR helpers defined (RidgeCV + NSE + shuffled K-fold, upstream-aligned)


## 3. Load HFA from disk

Reads every `hfa_*.pkl` file in `<HFA_CACHE_DIR>/<analysis_band>/` into `all_session_hfa`, sorted by subject/session.


In [31]:
import pickle
from pathlib import Path

band = CONFIG['analysis_band']
band_dir = Path(HFA_CACHE_DIR) / band
cache_files = sorted(band_dir.glob('hfa_*.pkl'))
print(f'Loading band {band!r} from {band_dir}')
print(f'Found {len(cache_files)} cached pickles')

if not cache_files:
    raise FileNotFoundError(
        f'No cache files in {band_dir}. Run hipp_amyg_compute_hfa.ipynb to generate '
        f'the {band!r} band first, or change CONFIG[\'analysis_band\'] to one that exists.'
    )

all_session_hfa = []
for fp in cache_files:
    try:
        with open(fp, 'rb') as f:
            r = pickle.load(f)
        if r is None:
            continue
        all_session_hfa.append(r)
    except Exception as e:
        print(f'  [skip] {fp.name}: {e}')

all_session_hfa.sort(key=lambda r: (r['subject'], r['session']))
print(f'Loaded {len(all_session_hfa)} sessions for band {band!r}:')
for r in all_session_hfa:
    print(f'  {r["subject"]}/{r["session"]}  '
          f'src={len(r["src_chans"])}  tgt={len(r["tgt_chans"])}')


Loading band 'theta' from E:\SBCAT\outputs\hipp_amyg_HFA_CSA_v2\hfa_cache\theta
Found 28 cached pickles
Loaded 28 sessions for band 'theta':
  sub-1/sub-1_ses-1_ecephys+image  src=14  tgt=15
  sub-1/sub-1_ses-2_ecephys+image  src=15  tgt=15
  sub-11/sub-11_ses-1_ecephys+image  src=15  tgt=14
  sub-12/sub-12_ses-1_ecephys+image  src=8  tgt=14
  sub-12/sub-12_ses-2_ecephys+image  src=8  tgt=8
  sub-13/sub-13_ses-1_ecephys+image  src=14  tgt=14
  sub-15/sub-15_ses-1_ecephys+image  src=14  tgt=15
  sub-16/sub-16_ses-1_ecephys+image  src=14  tgt=15
  sub-17/sub-17_ses-1_ecephys+image  src=14  tgt=16
  sub-19/sub-19_ses-1_ecephys+image  src=14  tgt=14
  sub-19/sub-19_ses-2_ecephys+image  src=14  tgt=14
  sub-2/sub-2_ses-1_ecephys+image  src=12  tgt=15
  sub-21/sub-21_ses-1_ecephys+image  src=26  tgt=16
  sub-22/sub-22_ses-1_ecephys+image  src=12  tgt=12
  sub-22/sub-22_ses-2_ecephys+image  src=27  tgt=11
  sub-22/sub-22_ses-3_ecephys+image  src=27  tgt=11
  sub-26/sub-26_ses-1_ecephys+image 

## 4. RRR  Communication Subspace Analysis

Two epoch sub-runs per session, each with three configurations:

1. **Encoding-cat**: window 0-2 s post-encoding1, group by 1st-stim category, all trials.
2. **Delay-load**: window 0-2.5 s post-maintenance, group by load (1 vs 3), all trials.

Per sub-run x config:
- **CS**: source -> target, N=10 channels each (with replacement).
- **Within-source**: split source into two non-overlapping halves.
- **Reverse**: target -> source.


### 4a. Single-session RRR demo (run before the full sweep)

Times one session end-to-end across both sub-runs (encoding-cat, delay-load) and all three configs (CS / Within / Reverse) so you can sanity-check the pipeline and estimate full-sweep wall-clock before kicking off all sessions.


In [ ]:
# Demo: run ONE session through the upstream-aligned RRR pipeline with progress.
# Same session as the dim demo (sub-15/sub-15_ses-1) for paired comparison.
DEMO_SUBJECT = 'sub-15'
DEMO_SESSION = 'sub-15_ses-1_ecephys+image'
PROGRESS_EVERY = 5

demo_session = next((r for r in all_session_hfa
                      if r['subject'] == DEMO_SUBJECT and r['session'] == DEMO_SESSION),
                     None)
if demo_session is None:
    available = [(r['subject'], r['session']) for r in all_session_hfa]
    raise RuntimeError(f'{DEMO_SUBJECT}/{DEMO_SESSION} not found in all_session_hfa.\n'
                       f'Available: {available}')

print(f'Demo session: {DEMO_SUBJECT}/{DEMO_SESSION}  '
      f'(src={len(demo_session["src_chans"])} ch, tgt={len(demo_session["tgt_chans"])} ch)')

rng_local = np.random.default_rng(CONFIG['random_seed'] + hash(DEMO_SESSION) % 10000)


def _bootstrap_rrr_verbose(label, X_pool, Y_pool, n_chans,
                            n_iters, alpha_init_grid, rank_grid, n_folds,
                            rng, sampling='replace'):
    """Same math as bootstrap_rrr; prints status every PROGRESS_EVERY iters."""
    rank_grid = np.asarray(rank_grid, dtype=int)
    n_x_total, n_y_total = X_pool.shape[1], Y_pool.shape[1]

    cs_dim     = np.full(n_iters, np.nan)
    best_nse   = np.full(n_iters, np.nan)
    nse_curves = np.full((n_iters, len(rank_grid)), np.nan)
    t_start = time.time()
    for it in range(n_iters):
        if sampling == 'disjoint':
            half = n_x_total // 2
            if half < 1:
                continue
            perm = rng.permutation(n_x_total)
            half_a, half_b = perm[:half], perm[half:half + half]
            x_idx = rng.choice(half_a, n_chans, replace=True)
            y_idx = rng.choice(half_b, n_chans, replace=True)
        else:
            x_idx = rng.choice(n_x_total, n_chans, replace=True)
            y_idx = rng.choice(n_y_total, n_chans, replace=True)
        Xb, Yb = X_pool[:, x_idx], Y_pool[:, y_idx]
        try:
            nse_grid = cv_rrr_nse(Xb, Yb, rank_grid, alpha_init_grid,
                                    n_folds=n_folds, rng=rng, iter_idx=it)
            cs, mn, _ = select_dim_1se(nse_grid, rank_grid)
            cs_dim[it]     = cs
            nse_curves[it] = mn
            if np.any(np.isfinite(mn)):
                best_nse[it] = float(np.nanmin(mn))
        except Exception:
            continue
        done = it + 1
        if done % PROGRESS_EVERY == 0 or done == n_iters:
            elapsed = time.time() - t_start
            rate = done / elapsed if elapsed > 0 else 0
            eta = (n_iters - done) / rate if rate > 0 else float('nan')
            mean_r2 = float(1.0 - np.nanmean(best_nse))
            print(f'    [{label}] iter {done:>3}/{n_iters}  '
                  f'mean CS dim {np.nanmean(cs_dim):.2f}  '
                  f'mean R\u00b2 {mean_r2:.3f}  '
                  f'elapsed {elapsed:.1f}s  ETA {eta:.1f}s')
    r2_curves = 1.0 - nse_curves
    return {'cs_dim': cs_dim, 'best_nse': best_nse, 'best_r2': 1.0 - best_nse,
            'nse_curves': nse_curves, 'r2_curves': r2_curves,
            'rank_grid': rank_grid, 'elapsed': time.time() - t_start}


sub_run_specs = [
    ('encoding_cat', 'encoding1',   dict(mode='category', keep=CONFIG['category_keep'])),
    ('delay_load',   'maintenance', dict(mode='load')),
]

demo_results = {}
t0_total = time.time()
rank_grid = np.arange(1, CONFIG['rrr_n_chans'] + 1)

for sub_run, ev_key, group_kwargs in sub_run_specs:
    print(f'\n=== sub_run = {sub_run} ===')
    ev = demo_session['event_tensors'].get(ev_key)
    if ev is None or ev['src'].shape[0] == 0:
        print(f'  [skip] no data for event {ev_key}')
        continue
    groups = get_trial_groups(ev['meta'], **group_kwargs)
    X_src = build_rrr_data_matrix(ev['src'], groups['groups'])
    Y_tgt = build_rrr_data_matrix(ev['tgt'], groups['groups'])
    print(f'  data matrices: X_src {X_src.shape}, Y_tgt {Y_tgt.shape}')

    sub_run_results = {}
    sub_run_results['cs'] = _bootstrap_rrr_verbose(
        f'{sub_run}/CS',
        X_src, Y_tgt, CONFIG['rrr_n_chans'],
        CONFIG['rrr_n_iters'], CONFIG['rrr_alpha_init_grid'], rank_grid,
        CONFIG['rrr_n_folds'], rng_local, sampling='replace',
    )
    sub_run_results['within'] = _bootstrap_rrr_verbose(
        f'{sub_run}/Within',
        X_src, X_src, CONFIG['rrr_n_chans'],
        CONFIG['rrr_n_iters'], CONFIG['rrr_alpha_init_grid'], rank_grid,
        CONFIG['rrr_n_folds'], rng_local, sampling='disjoint',
    )
    sub_run_results['reverse'] = _bootstrap_rrr_verbose(
        f'{sub_run}/Reverse',
        Y_tgt, X_src, CONFIG['rrr_n_chans'],
        CONFIG['rrr_n_iters'], CONFIG['rrr_alpha_init_grid'], rank_grid,
        CONFIG['rrr_n_folds'], rng_local, sampling='replace',
    )
    demo_results[sub_run] = sub_run_results

    print(f'  Summary [{sub_run}]:')
    for kind in ['cs', 'within', 'reverse']:
        r = sub_run_results[kind]
        print(f'    {kind:>8s}: CS dim={np.nanmean(r["cs_dim"]):.2f}, '
              f'best R\u00b2={np.nanmean(r["best_r2"]):.3f} '
              f'({r["elapsed"]:.1f}s)')

total_elapsed = time.time() - t0_total
print(f'\nTotal demo elapsed: {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)')


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sub_run_labels = [('encoding_cat', 'Encoding (category)'),
                   ('delay_load',   'Delay (load)')]
for i, (sub_run, sr_label) in enumerate(sub_run_labels):
    ax = axes[i]
    if sub_run not in demo_results:
        ax.set_title(f'{sr_label}  (no data)')
        continue
    for kind, color in [('cs', '#4C72B0'), ('within', '#888888'),
                         ('reverse', '#DD8452')]:
        r = demo_results[sub_run][kind]
        curves = r['r2_curves']
        if curves.size == 0:
            continue
        mean = np.nanmean(curves, axis=0)
        n_valid = np.maximum(np.sum(~np.isnan(curves), axis=0), 1)
        sem = np.nanstd(curves, axis=0, ddof=1) / np.sqrt(n_valid)
        xs = r['rank_grid']
        ax.plot(xs, mean, '-o', color=color, lw=2, markersize=4,
                label=f'{kind.upper()}  CS dim={np.nanmean(r["cs_dim"]):.1f}')
        ax.fill_between(xs, mean - sem, mean + sem, color=color, alpha=0.2)
    ax.axhline(0, color='k', ls='--', lw=0.5, alpha=0.4)
    ax.set_xlabel('Rank')
    ax.set_ylabel('CV R\u00b2  (= 1 \u2212 NSE)')
    ax.set_title(f'{sr_label}  ({DEMO_SUBJECT}/{DEMO_SESSION.split("_ecephys")[0]})')
    ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'rrr_demo_curves.png', dpi=130, bbox_inches='tight')
plt.show()


_n_sess = len(all_session_hfa)
_n_workers = CONFIG['n_workers']
_batches = int(np.ceil(_n_sess / _n_workers))
print(f'\nFull-sweep wall-clock estimate (this-session pace): '
      f'~{total_elapsed * _batches / 60:.0f} min  '
      f'({_batches} batch(es) of up to {_n_workers} parallel sessions)')
print('If timing looks acceptable, run the next cell to do all sessions.')


Demo session: sub-15/sub-15_ses-1_ecephys+image  (src=14 ch, tgt=15 ch)

=== sub_run = encoding_cat ===
  data matrices: X_src (4000, 14), Y_tgt (4000, 15)
    [encoding_cat/CS] iter   5/100  mean CS dim 3.80  mean R² 0.276  elapsed 5.0s  ETA 95.4s
    [encoding_cat/CS] iter  10/100  mean CS dim 3.60  mean R² 0.261  elapsed 10.0s  ETA 90.0s
    [encoding_cat/CS] iter  15/100  mean CS dim 3.33  mean R² 0.255  elapsed 14.7s  ETA 83.2s
    [encoding_cat/CS] iter  20/100  mean CS dim 3.20  mean R² 0.256  elapsed 19.5s  ETA 78.1s
    [encoding_cat/CS] iter  25/100  mean CS dim 3.08  mean R² 0.246  elapsed 24.4s  ETA 73.2s
    [encoding_cat/CS] iter  30/100  mean CS dim 3.10  mean R² 0.246  elapsed 29.2s  ETA 68.1s
    [encoding_cat/CS] iter  35/100  mean CS dim 3.09  mean R² 0.244  elapsed 34.1s  ETA 63.3s
    [encoding_cat/CS] iter  40/100  mean CS dim 3.08  mean R² 0.245  elapsed 39.0s  ETA 58.4s
    [encoding_cat/CS] iter  45/100  mean CS dim 3.11  mean R² 0.245  elapsed 43.6s  ETA 53.3s

In [ ]:
def run_rrr_for_session(session_result, cfg, sub_run, rng):
    """Run all 3 configs (CS / Within / Reverse) for one session and one sub_run.
    Sample n_chans from each pool with replacement; rank grid 1..n_chans."""
    if sub_run == 'encoding_cat':
        ev = session_result['event_tensors'].get('encoding1')
        groups = get_trial_groups(ev['meta'], mode='category', keep=cfg['category_keep'])
    elif sub_run == 'delay_load':
        ev = session_result['event_tensors'].get('maintenance')
        groups = get_trial_groups(ev['meta'], mode='load')
    else:
        raise ValueError(sub_run)

    if ev is None or ev['src'].shape[0] == 0:
        return None

    X_src = build_rrr_data_matrix(ev['src'], groups['groups'])
    Y_tgt = build_rrr_data_matrix(ev['tgt'], groups['groups'])

    n_chan_src = X_src.shape[1]
    n_chan_tgt = Y_tgt.shape[1]
    if n_chan_src < 2 or n_chan_tgt < 2 or X_src.shape[0] < cfg['rrr_n_folds']:
        return None

    rank_grid = np.arange(1, cfg['rrr_n_chans'] + 1)

    cs = bootstrap_rrr(
        X_src, Y_tgt, cfg['rrr_n_chans'],
        cfg['rrr_n_iters'], cfg['rrr_alpha_init_grid'],
        rank_grid, cfg['rrr_n_folds'], rng, sampling='replace',
    )
    within = bootstrap_rrr(
        X_src, X_src, cfg['rrr_n_chans'],
        cfg['rrr_n_iters'], cfg['rrr_alpha_init_grid'],
        rank_grid, cfg['rrr_n_folds'], rng, sampling='disjoint',
    )
    reverse = bootstrap_rrr(
        Y_tgt, X_src, cfg['rrr_n_chans'],
        cfg['rrr_n_iters'], cfg['rrr_alpha_init_grid'],
        rank_grid, cfg['rrr_n_folds'], rng, sampling='replace',
    )
    return {'cs': cs, 'within': within, 'reverse': reverse,
            'n_chan_src': n_chan_src, 'n_chan_tgt': n_chan_tgt,
            'X_shape': X_src.shape, 'Y_shape': Y_tgt.shape}


def _rrr_one(session_result, cfg):
    out = {'subject': session_result['subject'], 'session': session_result['session']}
    rng = np.random.default_rng(cfg['random_seed'] + hash(session_result['session']) % 10000)
    for sub_run in ['encoding_cat', 'delay_load']:
        try:
            res = run_rrr_for_session(session_result, cfg, sub_run, rng)
        except Exception as e:
            res = None
            print(f'  [{sub_run} error] {session_result["subject"]}/{session_result["session"]}: {e}')
        if res is None:
            for kind in ['cs', 'within', 'reverse']:
                for stat in ['cs_dim_mean', 'best_r2_mean']:
                    out[f'{sub_run}_{kind}_{stat}'] = np.nan
            continue
        for kind in ['cs', 'within', 'reverse']:
            r = res[kind]
            out[f'{sub_run}_{kind}_cs_dim_mean'] = float(np.nanmean(r['cs_dim']))
            out[f'{sub_run}_{kind}_best_r2_mean'] = float(np.nanmean(r['best_r2']))
        out[f'{sub_run}_n_chan_src'] = res['n_chan_src']
        out[f'{sub_run}_n_chan_tgt'] = res['n_chan_tgt']
    return out


t0 = time.time()
print(f'RRR sweep: {len(all_session_hfa)} sessions x 2 sub-runs x 3 configs, '
      f'iters={CONFIG["rrr_n_iters"]}, n_chans={CONFIG["rrr_n_chans"]}, '
      f'{CONFIG["n_workers"]} workers, loky)')
rrr_rows = Parallel(n_jobs=CONFIG['n_workers'], backend='loky', verbose=10)(
    delayed(_rrr_one)(r, CONFIG) for r in all_session_hfa
)
for r in rrr_rows:
    print(f'  [done] {r["subject"]}/{r["session"]}: '
          f"enc-CS dim={r.get('encoding_cat_cs_cs_dim_mean', np.nan):.2f}, "
          f"delay-CS dim={r.get('delay_load_cs_cs_dim_mean', np.nan):.2f}")
rrr_df = pd.DataFrame(rrr_rows)
rrr_df.to_csv(OUT_DIR / 'rrr_summary.csv', index=False)
print(f'\nDone: {len(rrr_df)} sessions in {time.time()-t0:.0f}s -> {OUT_DIR/"rrr_summary.csv"}')
rrr_df.head()


In [ ]:
# RRR group-level paired tests + plots, per sub-run
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
sub_runs = [('encoding_cat', 'Encoding (category)'), ('delay_load', 'Delay (load)')]
metrics = [('cs_dim_mean', 'CS dim'), ('best_r2_mean', 'peak R²')]

for ri, (sr, sr_label) in enumerate(sub_runs):
    for ci, (metric, metric_label) in enumerate(metrics):
        ax = axes[ri, ci]
        cols = [f'{sr}_{kind}_{metric}' for kind in ['cs', 'within', 'reverse']]
        labels_kinds = ['CS', 'Within', 'Reverse']
        valid = rrr_df.dropna(subset=cols)
        if len(valid) == 0:
            ax.set_title(f'{sr_label} — {metric_label}\n(no data)')
            continue
        data = [valid[c] for c in cols]
        parts = ax.boxplot(data, labels=labels_kinds, showmeans=True, patch_artist=True)
        for patch, color in zip(parts['boxes'], ['#4C72B0', '#888888', '#DD8452']):
            patch.set_facecolor(color); patch.set_alpha(0.6)
        for _, r in valid.iterrows():
            ax.plot([1, 2, 3], [r[c] for c in cols], 'k-', alpha=0.2, lw=0.6)
            ax.plot([1, 2, 3], [r[c] for c in cols], 'ko', alpha=0.4, markersize=3)
        ax.set_ylabel(metric_label)
        ax.set_title(f'{sr_label} — {metric_label} (n={len(valid)})')

plt.tight_layout()
plt.savefig(OUT_DIR / 'rrr_group.png', dpi=130, bbox_inches='tight')
plt.show()

# Paired Wilcoxon: CS vs Within for each sub-run
print('\n=== Group paired tests (CS vs Within) ===')
for sr, sr_label in sub_runs:
    a_col = f'{sr}_cs_cs_dim_mean'
    b_col = f'{sr}_within_cs_dim_mean'
    valid = rrr_df.dropna(subset=[a_col, b_col])
    if len(valid) >= 5:
        stat, p = scistats.wilcoxon(valid[a_col], valid[b_col], zero_method='wilcox')
        print(f'  [{sr_label}] CS dim vs Within dim: n={len(valid)}, W={stat:.1f}, p={p:.4g}; '
              f'medians {valid[a_col].median():.2f} vs {valid[b_col].median():.2f}')

    a_col = f'{sr}_cs_best_r2_mean'
    b_col = f'{sr}_within_best_r2_mean'
    valid = rrr_df.dropna(subset=[a_col, b_col])
    if len(valid) >= 5:
        stat, p = scistats.wilcoxon(valid[a_col], valid[b_col], zero_method='wilcox')
        print(f'  [{sr_label}] peak R² CS vs Within: n={len(valid)}, W={stat:.1f}, p={p:.4g}; '
              f'medians {valid[a_col].median():.3f} vs {valid[b_col].median():.3f}')